# Fundamental EDA - Predicting Student Health Risk

This notebook explores the **Playground Series S6E7: Predicting Student Health Risk** dataset.

The goal is to understand target balance, missingness, feature behavior, categorical patterns, data-quality risks, and train/test drift before building a baseline model. The notebook is intentionally object-oriented: loading, schema inspection, chart rendering, diagnostics, and takeaway generation live in reusable classes.


# Context & Methods

## Key Assumptions

- The notebook must work on Kaggle, where input files usually live under `/kaggle/input/...`.
- The same notebook should also run locally against `dataset/` in this project folder.
- The task is treated as multi-class classification with `health_condition` as the target.
- Visualizations avoid non-standard plotting dependencies and use inline HTML/SVG for portability.

## Reading Guide

The analysis moves from broad checks to modeling implications:

1. **Data and schema** - confirm files, columns, and roles.
2. **Quality checks** - look for duplicate ids, overlap, missing values, and submission issues.
3. **Target and features** - inspect class balance and feature behavior.
4. **Drift and relationships** - compare train/test distributions and numeric correlations.
5. **Takeaways** - summarize what a baseline model should preserve.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import html, math
import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    def display(obj): print(str(obj)[:1200])

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")


# Data

The input resolver checks common Kaggle locations first and then falls back to local folders. This keeps the notebook reusable without path edits.


In [ ]:
@dataclass(frozen=True)
class EDAConfig:
    competition_slug: str = "playground-series-s6e7"
    local_dataset_dir: Path = Path("../dataset")
    target: str = "health_condition"
    id_column: str = "id"
    random_state: int = 42
    max_hist_rows: int = 120_000
    max_category_levels: int = 12
    svg_width: int = 980

class DataLoader:
    def __init__(self, config: EDAConfig): self.config = config
    def resolve_input_dir(self) -> Path:
        candidates = [
            Path("/kaggle/input/predicting-student-health-risk"),
            Path("/kaggle/input") / self.config.competition_slug,
            Path("/kaggle/input/playground-series-s6e7"),
            self.config.local_dataset_dir,
            Path("dataset"),
        ]
        for path in candidates:
            if (path / "train.csv").exists() and (path / "test.csv").exists(): return path
        kaggle_root = Path("/kaggle/input")
        if kaggle_root.exists():
            for train_path in sorted(kaggle_root.rglob("train.csv")):
                path = train_path.parent
                if (path / "test.csv").exists(): return path
        raise FileNotFoundError("Could not find train.csv and test.csv. Checked fixed paths and recursive /kaggle/input search: " + ", ".join(map(str, candidates)))
    def load(self):
        path = self.resolve_input_dir()
        return pd.read_csv(path / "train.csv"), pd.read_csv(path / "test.csv"), pd.read_csv(path / "sample_submission.csv"), path

class NotebookDisplay:
    @staticmethod
    def html(content: str):
        display(HTML(content)) if HTML else print(content[:1200])
    @staticmethod
    def card(title: str, body: str):
        style = "border:1px solid #dbe3ef;border-radius:8px;padding:16px 18px;margin:12px 0;background:#ffffff;box-shadow:0 1px 2px rgba(15,23,42,.05);"
        NotebookDisplay.html(f"<div style='{style}'><div style='font-size:17px;font-weight:700;color:#0f172a;margin-bottom:8px'>{html.escape(title)}</div><div style='font-size:14px;line-height:1.55;color:#334155'>{body}</div></div>")

config = EDAConfig()
train, test, sample_submission, input_dir = DataLoader(config).load()
NotebookDisplay.card("Input resolved", f"Loaded data from <code>{html.escape(str(input_dir))}</code>. Train rows: <b>{len(train):,}</b>, test rows: <b>{len(test):,}</b>, submission rows: <b>{len(sample_submission):,}</b>.")


In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)
display(train.head())
display(test.head())
display(sample_submission.head())


## Schema Overview

The schema check separates identifiers, target, numeric features, and categorical features. This is the first pass at understanding what a baseline model will need to handle.


In [ ]:
class SchemaInspector:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, test: pd.DataFrame):
        self.config, self.train, self.test = config, train, test
    @property
    def feature_columns(self): return [c for c in self.train.columns if c not in {self.config.id_column, self.config.target}]
    @property
    def numeric_columns(self): return [c for c in self.feature_columns if pd.api.types.is_numeric_dtype(self.train[c])]
    @property
    def categorical_columns(self): return [c for c in self.feature_columns if c not in self.numeric_columns]
    def summary(self):
        rows=[]
        for c in self.train.columns:
            rows.append({
                "column": c,
                "role": "id" if c == self.config.id_column else "target" if c == self.config.target else "numeric feature" if pd.api.types.is_numeric_dtype(self.train[c]) else "categorical feature",
                "train_dtype": str(self.train[c].dtype),
                "test_dtype": str(self.test[c].dtype) if c in self.test.columns else "-",
                "train_missing_%": self.train[c].isna().mean() * 100,
                "test_missing_%": self.test[c].isna().mean() * 100 if c in self.test.columns else np.nan,
                "train_unique": self.train[c].nunique(dropna=True),
                "test_unique": self.test[c].nunique(dropna=True) if c in self.test.columns else np.nan,
            })
        return pd.DataFrame(rows)

schema = SchemaInspector(config, train, test)
schema_summary = schema.summary()
display(schema_summary)
print("Numeric features:", schema.numeric_columns)
print("Categorical features:", schema.categorical_columns)


## Data Integrity Checks

Before looking at patterns, check whether the files line up: ids should be unique, train and test ids should not overlap, and the sample submission should match the test ids.


In [ ]:
class DataQualityAuditor:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, test: pd.DataFrame, sample_submission: pd.DataFrame):
        self.config = config
        self.train = train
        self.test = test
        self.sample_submission = sample_submission

    def summary(self) -> pd.DataFrame:
        id_col = self.config.id_column
        train_ids = set(self.train[id_col])
        test_ids = set(self.test[id_col])
        sample_ids = set(self.sample_submission[id_col])
        feature_columns = [c for c in self.train.columns if c != self.config.target]
        return pd.DataFrame([
            {"check": "train rows", "value": len(self.train)},
            {"check": "test rows", "value": len(self.test)},
            {"check": "sample submission rows", "value": len(self.sample_submission)},
            {"check": "duplicate train ids", "value": int(self.train[id_col].duplicated().sum())},
            {"check": "duplicate test ids", "value": int(self.test[id_col].duplicated().sum())},
            {"check": "train/test id overlap", "value": len(train_ids & test_ids)},
            {"check": "test ids missing from submission", "value": len(test_ids - sample_ids)},
            {"check": "submission ids not in test", "value": len(sample_ids - test_ids)},
            {"check": "duplicate full train rows", "value": int(self.train.duplicated().sum())},
            {"check": "constant feature columns", "value": int(sum(self.train[c].nunique(dropna=False) <= 1 for c in feature_columns))},
        ])

    def column_role_counts(self, schema_summary: pd.DataFrame) -> pd.Series:
        return schema_summary["role"].value_counts()

quality = DataQualityAuditor(config, train, test, sample_submission)
quality_summary = quality.summary()
display(quality_summary)
NotebookDisplay.card(
    "Data integrity summary",
    "<br>".join(f"<b>{html.escape(str(row.check))}</b>: {int(row.value):,}" for row in quality_summary.itertuples(index=False))
)


# Results

## Target Distribution

Target balance is the first modeling constraint. If one class dominates, plain accuracy can be misleading and validation should include per-class diagnostics.


In [ ]:
class SVGCharts:
    def __init__(self, width=980):
        self.width = width
        self.palette = {
            "at-risk": "#3B82F6",
            "fit": "#10B981",
            "unhealthy": "#EF4444",
            "Missing": "#F59E0B",
            "train_missing_%": "#3B82F6",
            "test_missing_%": "#64748B",
            "majority": "#64748B",
            "categorical_rules": "#3B82F6",
        }
        self.ink = "#0f172a"
        self.muted = "#64748b"
        self.border = "#dbe3ef"
        self.panel = "#ffffff"

    def wrap(self, title, subtitle, svg):
        return (
            f"<div style=\"font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;"
            f"border:1px solid {self.border};border-radius:8px;padding:16px;margin:14px 0;background:{self.panel}\">"
            f"<div style='font-size:18px;font-weight:700;color:{self.ink}'>{html.escape(title)}</div>"
            f"<div style='font-size:13px;color:{self.muted};margin:4px 0 12px 0'>{html.escape(subtitle)}</div>{svg}</div>"
        )

    def bar(self, data, title, subtitle, suffix=""):
        values = data.fillna(0).astype(float)
        maxv = max(values.abs().max(), 1)
        left, row_h, top = 230, 34, 18
        chart_w = self.width - left - 90
        parts = []
        for i, (label, value) in enumerate(values.items()):
            y = top + i * row_h
            w = chart_w * abs(value) / maxv
            color = self.palette.get(str(label), "#3B82F6")
            parts += [
                f'<text x="0" y="{y+20}" font-size="12" fill="{self.ink}">{html.escape(str(label))}</text>',
                f'<rect x="{left}" y="{y+6}" width="{w:.1f}" height="18" rx="3" fill="{color}" opacity=".88"></rect>',
                f'<text x="{left+w+8:.1f}" y="{y+20}" font-size="12" fill="{self.ink}">{value:,.2f}{html.escape(suffix)}</text>',
            ]
        return self.wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {top*2+row_h*len(values)}" width="100%">{"".join(parts)}</svg>')

    def grouped_bar(self, frame, title, subtitle, suffix="%"):
        left, top, group_h, bar_h = 230, 24, 76, 15
        chart_w = self.width - left - 90
        maxv = max(float(frame.max().max()), 1)
        parts = []
        for r, (label, row) in enumerate(frame.iterrows()):
            base = top + r * group_h
            parts.append(f'<text x="0" y="{base+15}" font-size="12" font-weight="700" fill="{self.ink}">{html.escape(str(label))}</text>')
            for j, col in enumerate(frame.columns):
                value = float(row[col])
                y = base + 24 + j * (bar_h + 4)
                w = chart_w * value / maxv
                color = self.palette.get(str(col), "#3B82F6")
                parts += [
                    f'<rect x="{left}" y="{y}" width="{w:.1f}" height="{bar_h}" rx="3" fill="{color}" opacity=".88"></rect>',
                    f'<text x="{left+w+7:.1f}" y="{y+12}" font-size="11" fill="{self.ink}">{html.escape(str(col))}: {value:,.1f}{html.escape(suffix)}</text>',
                ]
        return self.wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {top*2+group_h*len(frame)}" width="100%">{"".join(parts)}</svg>')

    def histogram_grid(self, frame, columns, title, subtitle, bins=24):
        columns = list(columns)
        cw, ch, gx, gy, per = 285, 120, 28, 46, 3
        height = math.ceil(len(columns) / per) * (ch + gy) + 18
        parts = []
        for i, c in enumerate(columns):
            values = frame[c].dropna().to_numpy()
            if len(values) == 0:
                continue
            counts, edges = np.histogram(values, bins=bins)
            maxc = max(counts.max(), 1)
            x0, y0 = (i % per) * (cw + gx), (i // per) * (ch + gy) + 20
            parts.append(f'<text x="{x0}" y="{y0-6}" font-size="12" font-weight="700" fill="{self.ink}">{html.escape(c)}</text>')
            for b, count in enumerate(counts):
                bw = cw / bins - 1
                bh = (ch - 24) * count / maxc
                x = x0 + b * (cw / bins)
                y = y0 + ch - 20 - bh
                parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" fill="#3B82F6" opacity=".78"></rect>')
            parts += [
                f'<line x1="{x0}" y1="{y0+ch-20}" x2="{x0+cw}" y2="{y0+ch-20}" stroke="#CBD5E1"/>',
                f'<text x="{x0}" y="{y0+ch-4}" font-size="10" fill="{self.muted}">{edges[0]:,.1f}</text>',
                f'<text x="{x0+cw-52}" y="{y0+ch-4}" font-size="10" fill="{self.muted}">{edges[-1]:,.1f}</text>',
            ]
        return self.wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {height}" width="100%">{"".join(parts)}</svg>')

    def heatmap(self, matrix, title, subtitle, suffix="", signed=True):
        left, top, cell_h = 210, 92, 34
        cell_w = min(104, max(66, (self.width - left - 40) / max(len(matrix.columns), 1)))
        height = top + cell_h * len(matrix.index) + 34
        values = matrix.to_numpy(dtype=float)
        max_abs = max(float(np.nanmax(np.abs(values))), 1e-9) if signed else max(float(np.nanmax(values)), 1e-9)
        parts = []
        for j, col in enumerate(matrix.columns):
            x = left + j * cell_w + cell_w / 2
            parts.append(f'<text transform="translate({x:.1f},78) rotate(-35)" text-anchor="end" font-size="11" fill="{self.ink}">{html.escape(str(col))}</text>')
        for i, (idx, row) in enumerate(matrix.iterrows()):
            y = top + i * cell_h
            parts.append(f'<text x="0" y="{y+22}" font-size="11" fill="{self.ink}">{html.escape(str(idx))}</text>')
            for j, value in enumerate(row):
                value = float(value)
                alpha = min(abs(value) / max_abs, 1) if signed else min(value / max_abs, 1)
                if signed and value < 0:
                    color = "254,202,202"
                else:
                    color = "37,99,235"
                parts += [
                    f'<rect x="{left + j*cell_w}" y="{y}" width="{cell_w-2}" height="{cell_h-2}" rx="4" fill="rgba({color},{0.16+alpha*.72:.2f})"></rect>',
                    f'<text x="{left+j*cell_w+cell_w/2}" y="{y+21}" text-anchor="middle" font-size="10" fill="{self.ink}">{value:.2f}{html.escape(suffix)}</text>',
                ]
        return self.wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {height}" width="100%">{"".join(parts)}</svg>')
charts = SVGCharts(config.svg_width)


In [ ]:
target_counts = train[config.target].value_counts()
target_percent = train[config.target].value_counts(normalize=True).mul(100)
display(pd.DataFrame({"rows": target_counts, "share_%": target_percent.round(3)}))
NotebookDisplay.html(charts.bar(target_percent, "Target class distribution", "Share of rows by health condition in the training data.", "%"))


**Observation.** A dominant majority class would make a naive accuracy baseline look deceptively strong. Validation should include per-class recall or a confusion matrix once modeling starts.


## Feature-Target Strength Ranking

This section ranks features by simple dependency-free signal scores. Numeric features use an eta-squared style target separation score, categorical features use Cramer's V, and missing indicators are scored as binary categorical features.


In [ ]:
class FeatureTargetStrengthAnalyzer:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, numeric_columns: list[str], categorical_columns: list[str]):
        self.config = config
        self.train = train
        self.numeric_columns = numeric_columns
        self.categorical_columns = categorical_columns

    def numeric_eta_squared(self, column: str) -> float:
        data = self.train[[column, self.config.target]].dropna()
        if data.empty:
            return 0.0
        overall_mean = data[column].mean()
        total_ss = ((data[column] - overall_mean) ** 2).sum()
        if total_ss == 0:
            return 0.0
        between_ss = data.groupby(self.config.target)[column].agg(lambda s: len(s) * (s.mean() - overall_mean) ** 2).sum()
        return float(between_ss / total_ss)

    @staticmethod
    def cramers_v(feature: pd.Series, target: pd.Series) -> float:
        table = pd.crosstab(feature.fillna("Missing"), target)
        if table.empty or min(table.shape) < 2:
            return 0.0
        observed = table.to_numpy(dtype=float)
        expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / observed.sum()
        chi2 = np.divide((observed - expected) ** 2, expected, out=np.zeros_like(observed), where=expected > 0).sum()
        n = observed.sum()
        return float(np.sqrt((chi2 / n) / max(min(table.shape) - 1, 1)))

    def rank(self) -> pd.DataFrame:
        rows = []
        for column in self.numeric_columns:
            rows.append({"feature": column, "signal_type": "numeric eta^2", "score": self.numeric_eta_squared(column), "missing_%": self.train[column].isna().mean() * 100})
        for column in self.categorical_columns:
            rows.append({"feature": column, "signal_type": "categorical Cramer's V", "score": self.cramers_v(self.train[column], self.train[self.config.target]), "missing_%": self.train[column].isna().mean() * 100})
        for column in self.numeric_columns + self.categorical_columns:
            if self.train[column].isna().any():
                rows.append({"feature": f"{column}_is_missing", "signal_type": "missing Cramer's V", "score": self.cramers_v(self.train[column].isna(), self.train[self.config.target]), "missing_%": self.train[column].isna().mean() * 100})
        return pd.DataFrame(rows).sort_values("score", ascending=False)

feature_strength = FeatureTargetStrengthAnalyzer(config, train, schema.numeric_columns, schema.categorical_columns).rank()
display(feature_strength.round(5))
NotebookDisplay.html(charts.bar(
    feature_strength.head(18).set_index("feature")["score"],
    "Top feature-target signal scores",
    "Dependency-free ranking: eta-squared for numeric columns, Cramer's V for categorical and missing indicators.",
))


## Feature Importance

Feature-target scores are univariate. This block adds a small model-based feature importance view when `scikit-learn` is available. If the environment does not provide `sklearn`, the notebook falls back to the dependency-free signal ranking above.


In [ ]:
class FeatureImportanceAnalyzer:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, feature_columns: list[str], categorical_columns: list[str], feature_strength: pd.DataFrame):
        self.config = config
        self.train = train
        self.feature_columns = feature_columns
        self.categorical_columns = categorical_columns
        self.numeric_columns = [c for c in feature_columns if c not in categorical_columns]
        self.feature_strength = feature_strength

    def fallback_importance(self, reason: str) -> pd.DataFrame:
        base = self.feature_strength[~self.feature_strength["feature"].str.endswith("_is_missing")].copy()
        return base[["feature", "score"]].rename(columns={"score": "importance"}).assign(method="signal ranking fallback", note=reason)

    def encoded_frame(self, frame: pd.DataFrame) -> pd.DataFrame:
        encoded = pd.DataFrame(index=frame.index)
        for column in self.numeric_columns:
            encoded[column] = frame[column]
            encoded[f"{column}_is_missing"] = frame[column].isna().astype("int8")
            encoded[column] = encoded[column].fillna(encoded[column].median())
        for column in self.categorical_columns:
            encoded[column] = frame[column].fillna("Missing").astype("category").cat.codes.astype("int16")
            encoded[f"{column}_is_missing"] = frame[column].isna().astype("int8")
        return encoded

    def model_importance(self, sample_size: int = 160_000) -> tuple[pd.DataFrame, str]:
        try:
            from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
            from sklearn.inspection import permutation_importance
            from sklearn.metrics import accuracy_score, f1_score
            from sklearn.model_selection import train_test_split
        except Exception as exc:
            return self.fallback_importance(f"sklearn unavailable: {type(exc).__name__}"), f"fallback: sklearn unavailable ({type(exc).__name__})"

        data = self.train.sample(min(len(self.train), sample_size), random_state=self.config.random_state)
        X = self.encoded_frame(data[self.feature_columns])
        y = data[self.config.target]
        X_train, X_valid, y_train, y_valid = train_test_split(
            X, y, test_size=0.25, random_state=self.config.random_state, stratify=y
        )
        candidates = [
            ("extra_trees", ExtraTreesClassifier(n_estimators=180, min_samples_leaf=40, class_weight="balanced", n_jobs=-1, random_state=self.config.random_state)),
            ("random_forest", RandomForestClassifier(n_estimators=160, min_samples_leaf=60, class_weight="balanced_subsample", n_jobs=-1, random_state=self.config.random_state)),
        ]
        results = []
        status_parts = []
        for method_name, model in candidates:
            model.fit(X_train, y_train)
            prediction = model.predict(X_valid)
            accuracy = accuracy_score(y_valid, prediction)
            macro_f1 = f1_score(y_valid, prediction, average="macro")
            status_parts.append(f"{method_name}: accuracy={accuracy:.4f}, macro_f1={macro_f1:.4f}")
            built_in = pd.DataFrame({"feature": X.columns, "importance": model.feature_importances_})
            grouped = built_in.assign(raw_feature=built_in["feature"].str.replace(r"_is_missing$", "", regex=True)).groupby("raw_feature", as_index=False)["importance"].sum()
            grouped = grouped.rename(columns={"raw_feature": "feature"}).assign(method=f"{method_name} impurity")
            results.append(grouped)

            permutation = permutation_importance(model, X_valid, y_valid, scoring="f1_macro", n_repeats=3, random_state=self.config.random_state, n_jobs=-1)
            perm = pd.DataFrame({"feature": X.columns, "importance": permutation.importances_mean})
            perm = perm.assign(raw_feature=perm["feature"].str.replace(r"_is_missing$", "", regex=True)).groupby("raw_feature", as_index=False)["importance"].sum()
            perm = perm.rename(columns={"raw_feature": "feature"}).assign(method=f"{method_name} permutation")
            results.append(perm)

        importance = pd.concat(results, ignore_index=True)
        summary = importance.groupby("feature", as_index=False).agg(
            importance=("importance", "mean"),
            method_count=("method", "nunique"),
        ).sort_values("importance", ascending=False)
        return summary.assign(note="mean across impurity and permutation importance views"), "; ".join(status_parts)

importance_analyzer = FeatureImportanceAnalyzer(config, train, schema.feature_columns, schema.categorical_columns, feature_strength)
feature_importance, feature_importance_status = importance_analyzer.model_importance()
display(feature_importance.head(25).round(5))
NotebookDisplay.card("Feature importance status", html.escape(feature_importance_status))
NotebookDisplay.html(charts.bar(
    feature_importance.head(18).set_index("feature")["importance"].abs(),
    "Feature importance",
    "Mean model-based importance when sklearn is available; otherwise dependency-free signal ranking fallback.",
))


## Missing Values

Missingness matters twice: it can carry signal, and it can create train/test drift. The next views compare missing rates between train and test.


In [ ]:
class MissingnessAnalyzer:
    def __init__(self, config, train, test): self.config, self.train, self.test = config, train, test
    def summary(self):
        rows=[]
        for c in [x for x in self.train.columns if x != self.config.target]:
            rows.append({"column":c, "train_missing_%":self.train[c].isna().mean()*100, "test_missing_%":self.test[c].isna().mean()*100 if c in self.test.columns else np.nan, "delta_test_minus_train_pp":((self.test[c].isna().mean()-self.train[c].isna().mean())*100) if c in self.test.columns else np.nan})
        return pd.DataFrame(rows).sort_values("train_missing_%", ascending=False)
missing_summary = MissingnessAnalyzer(config, train, test).summary()
display(missing_summary)
NotebookDisplay.html(charts.grouped_bar(missing_summary.set_index("column")[["train_missing_%","test_missing_%"]].head(14), "Missing values by column", "Top columns by training missing rate, compared with test missing rate."))


#### Missingness by Target Class

Overall missing rates are useful, but target-specific missingness can be even more important. If one class has systematically different missing values, missing indicators may be predictive.


In [ ]:
feature_columns = schema.numeric_columns + schema.categorical_columns
missing_by_target = train.groupby(config.target)[feature_columns].apply(lambda frame: frame.isna().mean().mul(100)).T
missing_by_target["spread_pp"] = missing_by_target.max(axis=1) - missing_by_target.min(axis=1)
missing_by_target = missing_by_target.sort_values("spread_pp", ascending=False)
display(missing_by_target.round(2))
NotebookDisplay.html(charts.heatmap(
    missing_by_target.drop(columns="spread_pp").round(2),
    "Missingness by target class",
    "Percent of missing values within each target class. Larger class spreads can become useful missing indicators.",
    "%",
    signed=False,
))


#### Missingness Signal

The previous table shows missing rates by class. This table flips the view: for each feature, compare target shares when the feature is missing versus present.


In [ ]:
class MissingnessSignalAnalyzer:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, feature_columns: list[str]):
        self.config = config
        self.train = train
        self.feature_columns = feature_columns
        self.base_share = train[config.target].value_counts(normalize=True).mul(100)

    def scan(self) -> pd.DataFrame:
        rows = []
        for column in self.feature_columns:
            missing_mask = self.train[column].isna()
            if not missing_mask.any():
                continue
            for state_name, mask in [("missing", missing_mask), ("present", ~missing_mask)]:
                shares = self.train.loc[mask, self.config.target].value_counts(normalize=True).mul(100)
                for target_value in self.base_share.index:
                    target_share = shares.get(target_value, 0.0)
                    rows.append({
                        "feature": column,
                        "state": state_name,
                        "target": target_value,
                        "rows": int(mask.sum()),
                        "target_share_%": target_share,
                        "lift_vs_global_pp": target_share - self.base_share[target_value],
                    })
        return pd.DataFrame(rows).sort_values("lift_vs_global_pp", key=lambda s: s.abs(), ascending=False)

missing_signal = MissingnessSignalAnalyzer(config, train, feature_columns).scan()
display(missing_signal.head(30).round(2))
NotebookDisplay.html(charts.bar(
    missing_signal.head(18).set_index(missing_signal.head(18)["feature"] + " " + missing_signal.head(18)["state"] + " -> " + missing_signal.head(18)["target"])["lift_vs_global_pp"].abs(),
    "Strongest target lifts from missingness",
    "Absolute percentage-point lift versus the global class share for missing/present states.",
    " pp",
))


**Observation.** Columns with visible missing rates should usually get explicit missing indicators in a modeling notebook. If train and test missing rates are similar, the pattern is less likely to be a split artifact.


## Numeric Feature Distributions

The histograms below show the global shape of numeric features. Skew, hard bounds, and unusual spikes are useful hints for preprocessing and model choice.


In [ ]:
hist_sample = train.sample(min(len(train), config.max_hist_rows), random_state=config.random_state)
numeric_summary = train[schema.numeric_columns].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
numeric_summary["missing_%"] = train[schema.numeric_columns].isna().mean().mul(100)
display(numeric_summary)
NotebookDisplay.html(charts.histogram_grid(hist_sample, schema.numeric_columns, "Numeric feature distributions", f"Histograms use up to {config.max_hist_rows:,} sampled training rows for speed."))


**Observation.** Tree-based models are a natural first baseline because these numeric columns have bounded ranges, missing values, and likely non-linear interactions.


## Numeric Features by Target

Class-conditional averages are a compact way to find features that may separate the target classes. This is not causal evidence; it is a modeling diagnostic.


In [ ]:
class TargetComparison:
    def __init__(self, config, train, numeric_columns, categorical_columns): self.config, self.train, self.numeric_columns, self.categorical_columns = config, train, numeric_columns, categorical_columns
    def numeric_mean_lift(self):
        grouped = self.train.groupby(self.config.target, dropna=False)[self.numeric_columns].mean(); overall = self.train[self.numeric_columns].mean()
        return grouped.div(overall).sub(1).mul(100).T
    def category_target_mix(self, column):
        table = pd.crosstab(self.train[column].fillna("Missing"), self.train[self.config.target], normalize="index").mul(100)
        return table.loc[table.sum(axis=1).sort_values(ascending=False).index]
comparison = TargetComparison(config, train, schema.numeric_columns, schema.categorical_columns)
numeric_mean_lift = comparison.numeric_mean_lift()
display(numeric_mean_lift.round(2))
NotebookDisplay.html(charts.heatmap(numeric_mean_lift.round(2), "Numeric mean lift by target class", "Each cell is percent difference from the overall feature mean. Positive values are above the full-train average.", "%"))


#### Numeric Percentile Profile

Means can hide distribution shifts. The percentile profile compares median and tail behavior across target classes for each numeric feature.


In [ ]:
class NumericProfile:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, numeric_columns: list[str]):
        self.config = config
        self.train = train
        self.numeric_columns = numeric_columns

    def percentile_table(self) -> pd.DataFrame:
        rows = []
        for column in self.numeric_columns:
            for target_value, frame in self.train.groupby(self.config.target):
                values = frame[column].dropna()
                rows.append({
                    "feature": column,
                    "target": target_value,
                    "p05": values.quantile(.05),
                    "p50": values.quantile(.50),
                    "p95": values.quantile(.95),
                    "iqr": values.quantile(.75) - values.quantile(.25),
                    "missing_%": frame[column].isna().mean() * 100,
                })
        return pd.DataFrame(rows)

numeric_profile = NumericProfile(config, train, schema.numeric_columns).percentile_table()
display(numeric_profile.round(3))
median_lift = numeric_profile.pivot(index="feature", columns="target", values="p50")
median_lift = median_lift.div(train[schema.numeric_columns].median(), axis=0).sub(1).mul(100)
NotebookDisplay.html(charts.heatmap(
    median_lift.round(2),
    "Numeric median lift by target class",
    "Percent difference between each class median and the full-train median.",
    "%",
))


**Observation.** Large positive or negative cells indicate features worth preserving as-is. Small cells do not mean the feature is useless because interactions can still be predictive.


## Categorical Feature Mix

For each categorical feature, the chart shows the target composition within each category level. This helps identify categories with different risk profiles and spots levels where missing values behave like a real segment.


In [ ]:
for column in schema.categorical_columns:
    mix = comparison.category_target_mix(column).head(config.max_category_levels)
    display(mix.round(2))
    NotebookDisplay.html(charts.grouped_bar(mix, f"Target mix within {column}", "Rows sum to 100%. Missing values are shown as their own level when present."))


#### Categorical Signal Summary

The table below highlights category levels where the target mix differs most from the global target mix. This is a quick way to spot strong categorical signals without reading every chart manually.


In [ ]:
class CategoricalSignalScanner:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, categorical_columns: list[str]):
        self.config = config
        self.train = train
        self.categorical_columns = categorical_columns
        self.base_share = train[config.target].value_counts(normalize=True).mul(100)

    def scan(self) -> pd.DataFrame:
        rows = []
        for column in self.categorical_columns:
            filled = self.train[column].fillna("Missing")
            counts = filled.value_counts()
            mix = pd.crosstab(filled, self.train[self.config.target], normalize="index").mul(100)
            for level, values in mix.iterrows():
                for target_value in values.index:
                    rows.append({
                        "feature": column,
                        "level": level,
                        "target": target_value,
                        "rows": int(counts[level]),
                        "target_share_%": values[target_value],
                        "lift_vs_global_pp": values[target_value] - self.base_share[target_value],
                    })
        return pd.DataFrame(rows).sort_values("lift_vs_global_pp", key=lambda s: s.abs(), ascending=False)

categorical_signal = CategoricalSignalScanner(config, train, schema.categorical_columns).scan()
display(categorical_signal.head(30).round(2))
minority_signal = categorical_signal[categorical_signal["target"].isin(["fit", "unhealthy"])].head(18)
NotebookDisplay.html(charts.bar(
    minority_signal.set_index(minority_signal["feature"] + " = " + minority_signal["level"].astype(str) + " -> " + minority_signal["target"])["lift_vs_global_pp"].abs(),
    "Strongest categorical lifts for minority classes",
    "Absolute percentage-point lift versus the global class share for fit and unhealthy.",
    " pp",
))


## Class-Focused EDA

Because the target is imbalanced, global averages can hide what separates the minority classes. This block focuses on `fit` and `unhealthy` directly.


In [ ]:
class ClassFocusedEDA:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, numeric_columns: list[str], categorical_signal: pd.DataFrame):
        self.config = config
        self.train = train
        self.numeric_columns = numeric_columns
        self.categorical_signal = categorical_signal

    def numeric_focus(self, target_value: str) -> pd.DataFrame:
        mask = self.train[self.config.target] == target_value
        rows = []
        for column in self.numeric_columns:
            target_values = self.train.loc[mask, column].dropna()
            rest_values = self.train.loc[~mask, column].dropna()
            rest_median = rest_values.median()
            rows.append({
                "feature": column,
                "target_median": target_values.median(),
                "rest_median": rest_median,
                "median_lift_%": ((target_values.median() / rest_median) - 1) * 100 if rest_median else np.nan,
                "target_p05": target_values.quantile(.05),
                "target_p95": target_values.quantile(.95),
            })
        return pd.DataFrame(rows).sort_values("median_lift_%", key=lambda s: s.abs(), ascending=False)

    def categorical_focus(self, target_value: str, n: int = 10) -> pd.DataFrame:
        return self.categorical_signal[self.categorical_signal["target"] == target_value].head(n)

class_focus = ClassFocusedEDA(config, train, schema.numeric_columns, categorical_signal)
for target_value in ["fit", "unhealthy"]:
    NotebookDisplay.card("Class focus: " + target_value, "Numeric median shifts and strongest categorical lifts against the global class share.")
    numeric_focus = class_focus.numeric_focus(target_value)
    categorical_focus = class_focus.categorical_focus(target_value)
    display(numeric_focus.round(3))
    display(categorical_focus.round(2))
    NotebookDisplay.html(charts.bar(
        numeric_focus.set_index("feature")["median_lift_%"].abs(),
        f"Numeric median shifts for {target_value}",
        "Absolute percent shift versus all other classes.",
        "%",
    ))
    NotebookDisplay.html(charts.bar(
        categorical_focus.set_index(categorical_focus["feature"] + " = " + categorical_focus["level"].astype(str))["lift_vs_global_pp"].abs(),
        f"Strongest categorical lifts for {target_value}",
        "Absolute percentage-point lift versus the global class share.",
        " pp",
    ))


**Observation.** Categorical features should be encoded with missing values preserved as an explicit category. This keeps potentially meaningful missingness available to the model.


## Train/Test Drift Scan

A quick drift scan compares numeric means and categorical distributions between train and test. Large shifts would make local validation less trustworthy.


In [ ]:
class DriftScanner:
    def __init__(self, config, train, test, numeric_columns, categorical_columns): self.config, self.train, self.test, self.numeric_columns, self.categorical_columns = config, train, test, numeric_columns, categorical_columns
    def numeric_drift(self):
        rows=[]
        for c in self.numeric_columns:
            train_mean=self.train[c].mean(); test_mean=self.test[c].mean(); pooled_std=pd.concat([self.train[c], self.test[c]], ignore_index=True).std()
            rows.append({"column":c, "train_mean":train_mean, "test_mean":test_mean, "standardized_mean_diff":(test_mean-train_mean)/pooled_std if pooled_std else 0})
        return pd.DataFrame(rows).sort_values("standardized_mean_diff", key=lambda s:s.abs(), ascending=False)
    def categorical_drift(self):
        rows=[]
        for c in self.categorical_columns:
            train_share=self.train[c].fillna("Missing").value_counts(normalize=True); test_share=self.test[c].fillna("Missing").value_counts(normalize=True); levels=train_share.index.union(test_share.index)
            tvd=0.5*(train_share.reindex(levels, fill_value=0)-test_share.reindex(levels, fill_value=0)).abs().sum(); rows.append({"column":c, "total_variation_distance":tvd})
        return pd.DataFrame(rows).sort_values("total_variation_distance", ascending=False)
scanner = DriftScanner(config, train, test, schema.numeric_columns, schema.categorical_columns)
numeric_drift = scanner.numeric_drift(); categorical_drift = scanner.categorical_drift()
display(numeric_drift); display(categorical_drift)
NotebookDisplay.html(charts.bar(numeric_drift.set_index("column")["standardized_mean_diff"].abs(), "Numeric train/test drift", "Absolute standardized mean difference. Smaller values suggest train and test are similarly distributed."))
NotebookDisplay.html(charts.bar(categorical_drift.set_index("column")["total_variation_distance"].mul(100), "Categorical train/test drift", "Total variation distance between train and test category shares.", "%"))


**Observation.** If drift is small, a standard stratified validation split should be a reasonable first validation setup. If drift is large in a feature, inspect whether missingness or category levels explain the shift.


## Feature Correlation Map

Correlation is only a linear diagnostic, but it quickly identifies redundant numeric features and pairs that may interact with the target. The heatmap uses the shared soft-green theme and rotated labels so longer feature names stay readable.


In [ ]:
correlation = train[schema.numeric_columns].corr(method="spearman")
display(correlation.round(3))
NotebookDisplay.html(charts.heatmap(correlation.round(2), "Spearman correlation between numeric features", "Rank correlation is robust enough for a quick monotonic relationship scan."))


## Simple Feature Ideas

The next small table creates a few transparent derived features for inspection only. These are candidates for a later modeling notebook, not guaranteed improvements.


In [ ]:
class FeatureSketcher:
    def __init__(self, frame): self.frame = frame
    def build(self):
        features = pd.DataFrame(index=self.frame.index)
        features["steps_per_exercise_min"] = self.frame["step_count"] / self.frame["exercise_duration"].replace(0, np.nan)
        features["calories_per_step"] = self.frame["calorie_expenditure"] / self.frame["step_count"].replace(0, np.nan)
        features["sleep_water_product"] = self.frame["sleep_duration"] * self.frame["water_intake"]
        features["bmi_heart_rate_product"] = self.frame["bmi"] * self.frame["heart_rate"]
        return features.replace([np.inf, -np.inf], np.nan)
sketched_features = FeatureSketcher(train).build()
sketch_summary = sketched_features.describe(percentiles=[.05,.25,.5,.75,.95]).T
sketch_summary["missing_%"] = sketched_features.isna().mean().mul(100)
display(sketch_summary)
NotebookDisplay.html(charts.histogram_grid(sketched_features.sample(min(len(sketched_features), config.max_hist_rows), random_state=config.random_state), sketched_features.columns, "Candidate derived feature distributions", "These features are simple ratios/products that may capture activity efficiency or combined health signals."))


# Modeling Bridge

This section turns EDA observations into modeling inputs. It checks a few high-signal interactions, creates a concrete feature plan, and summarizes the modeling hypotheses that should carry into the baseline notebook.


## Interaction EDA

The strongest signals are not isolated. These pairwise checks inspect whether combinations of stress, sleep, physical activity, and smoking/alcohol behavior produce more distinct target mixes.


In [ ]:
class InteractionEDA:
    def __init__(self, config: EDAConfig, train: pd.DataFrame):
        self.config = config
        self.train = train

    def target_mix(self, left: str, right: str, min_rows: int = 1_000) -> pd.DataFrame:
        frame = self.train[[left, right, self.config.target]].copy()
        frame[left] = frame[left].fillna("Missing")
        frame[right] = frame[right].fillna("Missing")
        frame["interaction"] = frame[left].astype(str) + " | " + frame[right].astype(str)
        counts = frame["interaction"].value_counts()
        valid = counts[counts >= min_rows].index
        table = pd.crosstab(frame.loc[frame["interaction"].isin(valid), "interaction"], frame.loc[frame["interaction"].isin(valid), self.config.target], normalize="index").mul(100)
        table["rows"] = counts.reindex(table.index)
        return table.sort_values(["fit", "unhealthy"], ascending=False)

interaction_pairs = [
    ("stress_level", "physical_activity_level"),
    ("stress_level", "sleep_quality"),
    ("sleep_quality", "smoking_alcohol"),
    ("physical_activity_level", "smoking_alcohol"),
]
interaction_eda = InteractionEDA(config, train)
interaction_tables = {}
for left, right in interaction_pairs:
    table = interaction_eda.target_mix(left, right)
    interaction_tables[f"{left} x {right}"] = table
    display(table.round(2).head(16))
    NotebookDisplay.html(charts.heatmap(
        table.drop(columns="rows").head(16).round(2),
        f"Target mix: {left} x {right}",
        "Rows are category combinations; cells show target share within each combination.",
        "%",
        signed=False,
    ))


## EDA-to-Model Feature Plan

This plan is intentionally concrete: it names which raw columns, missing indicators, and simple derived features should be carried into the first modeling notebook.


In [ ]:
class FeaturePlanBuilder:
    def __init__(self, schema: SchemaInspector, missing_summary: pd.DataFrame, feature_strength: pd.DataFrame):
        self.schema = schema
        self.missing_summary = missing_summary
        self.feature_strength = feature_strength

    def build(self) -> pd.DataFrame:
        rows = []
        strength = self.feature_strength.set_index("feature")["score"].to_dict()
        missing = self.missing_summary.set_index("column")["train_missing_%"].to_dict()
        for column in self.schema.numeric_columns:
            rows.append({"feature": column, "kind": "raw numeric", "include": True, "reason": "base numeric feature", "signal_score": strength.get(column, 0), "missing_%": missing.get(column, 0)})
            if missing.get(column, 0) > 0:
                rows.append({"feature": f"{column}_is_missing", "kind": "missing indicator", "include": True, "reason": "missingness may carry target signal", "signal_score": strength.get(f"{column}_is_missing", 0), "missing_%": missing.get(column, 0)})
        for column in self.schema.categorical_columns:
            rows.append({"feature": column, "kind": "categorical", "include": True, "reason": "preserve category and Missing level", "signal_score": strength.get(column, 0), "missing_%": missing.get(column, 0)})
            if missing.get(column, 0) > 0:
                rows.append({"feature": f"{column}_is_missing", "kind": "missing indicator", "include": True, "reason": "explicit missing flag", "signal_score": strength.get(f"{column}_is_missing", 0), "missing_%": missing.get(column, 0)})
        for feature, reason in {
            "steps_per_exercise_min": "activity intensity proxy; cap extreme denominator effects",
            "calories_per_step": "energy per movement proxy; cap high tail",
            "sleep_water_product": "combined recovery/hydration proxy",
            "bmi_heart_rate_product": "simple body-load interaction proxy",
        }.items():
            rows.append({"feature": feature, "kind": "derived numeric", "include": True, "reason": reason, "signal_score": np.nan, "missing_%": np.nan})
        return pd.DataFrame(rows).sort_values(["kind", "signal_score"], ascending=[True, False])

feature_plan = FeaturePlanBuilder(schema, missing_summary, feature_strength).build()
display(feature_plan.round(5))
NotebookDisplay.html(charts.bar(
    feature_plan.dropna(subset=["signal_score"]).head(18).set_index("feature")["signal_score"],
    "Planned features by signal score",
    "Raw and missing-indicator features planned for the first baseline.",
))


## Modeling Hypotheses Summary

These hypotheses are the handoff from EDA to modeling. They should be validated by cross-validation and leaderboard feedback, not treated as final truths.


In [ ]:
modeling_hypotheses = pd.DataFrame([
    {"hypothesis": "stress_level is the strongest categorical driver", "evidence": "Highest Cramer's V and large lifts for low/high stress", "modeling_action": "Keep as categorical; inspect interactions with activity and sleep"},
    {"hypothesis": "sleep_duration separates unhealthy from fit", "evidence": "Large negative lift for unhealthy and positive lift for fit", "modeling_action": "Keep raw numeric; add interaction checks but avoid aggressive binning first"},
    {"hypothesis": "physical_activity_level helps identify fit", "evidence": "Active level has large fit lift", "modeling_action": "Keep categorical and interact with stress_level"},
    {"hypothesis": "missingness is secondary but not free noise", "evidence": "BMI missing has visible target lift", "modeling_action": "Add missing indicators for all missing columns"},
    {"hypothesis": "train/test distribution looks close", "evidence": "Low numeric standardized mean differences and low categorical TVD", "modeling_action": "Use stratified CV as first validation setup"},
    {"hypothesis": "accuracy alone hides minority-class behavior", "evidence": "Majority baseline has high accuracy but weak macro metrics", "modeling_action": "Track macro F1, macro recall, and confusion matrix alongside official metric"},
])
display(modeling_hypotheses)
NotebookDisplay.card(
    "Modeling handoff",
    "<br>".join(f"<b>{html.escape(row.hypothesis)}</b>: {html.escape(row.modeling_action)}" for row in modeling_hypotheses.itertuples(index=False))
)


# Sanity Baseline

This is not the modeling solution. It is a quick validation anchor: compare a majority-class baseline with a tiny rule-based model learned only from categorical levels in a stratified validation split.


In [ ]:
class SanityBaseline:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, categorical_columns: list[str], validation_frac: float = 0.2):
        self.config = config
        self.train = train
        self.categorical_columns = categorical_columns
        self.validation_frac = validation_frac
        self.classes = list(train[config.target].value_counts().index)

    def split(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        validation = self.train.groupby(self.config.target, group_keys=False).sample(frac=self.validation_frac, random_state=self.config.random_state)
        train_part = self.train.drop(validation.index)
        return train_part, validation

    def fit_rules(self, train_part: pd.DataFrame, min_rows: int = 500, min_lift_pp: float = 4.0) -> list[dict]:
        base_share = train_part[self.config.target].value_counts(normalize=True).mul(100)
        majority_class = base_share.idxmax()
        rules = []
        for column in self.categorical_columns:
            filled = train_part[column].fillna("Missing")
            counts = filled.value_counts()
            mix = pd.crosstab(filled, train_part[self.config.target], normalize="index").mul(100)
            for level, row in mix.iterrows():
                if counts[level] < min_rows:
                    continue
                for target_value in row.index:
                    if target_value == majority_class:
                        continue
                    lift = row[target_value] - base_share[target_value]
                    if lift >= min_lift_pp:
                        rules.append({"column": column, "level": level, "prediction": target_value, "confidence_%": row[target_value], "lift_pp": lift, "rows": int(counts[level])})
        return sorted(rules, key=lambda r: (r["lift_pp"], r["confidence_%"]), reverse=True)

    def predict_with_rules(self, frame: pd.DataFrame, majority_class: str, rules: list[dict]) -> pd.Series:
        predictions = pd.Series(majority_class, index=frame.index, dtype=object)
        assigned = pd.Series(False, index=frame.index)
        for rule in rules:
            values = frame[rule["column"]].fillna("Missing")
            mask = (~assigned) & (values == rule["level"])
            predictions.loc[mask] = rule["prediction"]
            assigned.loc[mask] = True
        return predictions

    def metrics(self, actual: pd.Series, predicted: pd.Series) -> pd.DataFrame:
        rows = []
        for label in self.classes:
            tp = int(((actual == label) & (predicted == label)).sum())
            fp = int(((actual != label) & (predicted == label)).sum())
            fn = int(((actual == label) & (predicted != label)).sum())
            precision = tp / (tp + fp) if tp + fp else 0.0
            recall = tp / (tp + fn) if tp + fn else 0.0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
            rows.append({"class": label, "precision": precision, "recall": recall, "f1": f1})
        per_class = pd.DataFrame(rows)
        return pd.DataFrame([
            {"metric": "accuracy", "value": float((actual == predicted).mean())},
            {"metric": "macro_precision", "value": float(per_class["precision"].mean())},
            {"metric": "macro_recall", "value": float(per_class["recall"].mean())},
            {"metric": "macro_f1", "value": float(per_class["f1"].mean())},
        ])

    def evaluate(self) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        train_part, validation = self.split()
        majority_class = train_part[self.config.target].mode().iat[0]
        majority_predictions = pd.Series(majority_class, index=validation.index)
        rules = self.fit_rules(train_part)
        rule_predictions = self.predict_with_rules(validation, majority_class, rules)
        metrics = pd.concat([
            self.metrics(validation[self.config.target], majority_predictions).assign(model="majority"),
            self.metrics(validation[self.config.target], rule_predictions).assign(model="categorical_rules"),
        ], ignore_index=True)
        confusion = pd.crosstab(validation[self.config.target], rule_predictions, rownames=["actual"], colnames=["predicted"], dropna=False)
        return metrics[["model", "metric", "value"]], pd.DataFrame(rules), confusion, validation

baseline = SanityBaseline(config, train, schema.categorical_columns)
baseline_metrics, baseline_rules, baseline_confusion, baseline_validation = baseline.evaluate()
display(baseline_metrics.pivot(index="metric", columns="model", values="value").round(4))
display(baseline_rules.head(20).round(2))
display(baseline_confusion)
NotebookDisplay.html(charts.grouped_bar(
    baseline_metrics.pivot(index="metric", columns="model", values="value").mul(100),
    "Sanity baseline metrics",
    "Majority baseline versus a tiny categorical-rule model on a stratified validation split.",
    "%",
))
NotebookDisplay.html(charts.heatmap(
    baseline_confusion.astype(float),
    "Categorical-rule confusion matrix",
    "Rows are actual labels and columns are predictions on the validation split.",
    signed=False,
))


# Baseline Submission

This cell creates a valid baseline submission. On Kaggle, it writes to `/kaggle/working/baseline_submission.csv`; locally, it writes next to the notebook. It uses a small sklearn pipeline when available and falls back to the majority class otherwise.


In [ ]:
class BaselineSubmissionBuilder:
    def __init__(self, config: EDAConfig, train: pd.DataFrame, test: pd.DataFrame, sample_submission: pd.DataFrame, feature_columns: list[str], categorical_columns: list[str]):
        self.config = config
        self.train = train
        self.test = test
        self.sample_submission = sample_submission
        self.feature_columns = feature_columns
        self.categorical_columns = categorical_columns
        self.numeric_columns = [c for c in feature_columns if c not in categorical_columns]

    def build(self) -> tuple[pd.DataFrame, str]:
        try:
            from sklearn.compose import ColumnTransformer
            from sklearn.ensemble import ExtraTreesClassifier
            from sklearn.impute import SimpleImputer
            from sklearn.pipeline import Pipeline
            from sklearn.preprocessing import OneHotEncoder
        except Exception as exc:
            majority_class = self.train[self.config.target].mode().iat[0]
            submission = self.sample_submission.copy()
            submission[self.config.target] = majority_class
            return submission, f"majority fallback because sklearn is unavailable: {type(exc).__name__}"

        preprocessor = ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), self.numeric_columns),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=200)),
            ]), self.categorical_columns),
        ])
        model = ExtraTreesClassifier(
            n_estimators=260,
            min_samples_leaf=35,
            class_weight=None,
            n_jobs=-1,
            random_state=self.config.random_state,
        )
        pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
        pipeline.fit(self.train[self.feature_columns], self.train[self.config.target])
        submission = self.sample_submission.copy()
        submission[self.config.target] = pipeline.predict(self.test[self.feature_columns])
        return submission, "ExtraTrees baseline trained on full train data"

    @staticmethod
    def output_path() -> Path:
        output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
        return output_dir / "baseline_submission.csv"

submission_builder = BaselineSubmissionBuilder(config, train, test, sample_submission, schema.feature_columns, schema.categorical_columns)
baseline_submission, baseline_submission_status = submission_builder.build()
baseline_submission_path = submission_builder.output_path()
baseline_submission.to_csv(baseline_submission_path, index=False)

display(baseline_submission.head())
display(baseline_submission[config.target].value_counts(normalize=True).mul(100).rename("prediction_share_%").to_frame().round(2))
NotebookDisplay.card(
    "Baseline submission written",
    f"Status: <b>{html.escape(baseline_submission_status)}</b><br>Path: <code>{html.escape(str(baseline_submission_path))}</code><br>Rows: <b>{len(baseline_submission):,}</b>"
)


# Takeaways

The cell below writes a compact summary from the executed notebook state. It should update automatically if the dataset changes.


In [ ]:
class TakeawayWriter:
    def __init__(self, config, train, test, missing_summary, numeric_drift, categorical_drift): self.config, self.train, self.test, self.missing_summary, self.numeric_drift, self.categorical_drift = config, train, test, missing_summary, numeric_drift, categorical_drift
    def render(self):
        target_share = self.train[self.config.target].value_counts(normalize=True).mul(100)
        majority_class = target_share.idxmax(); majority_share = target_share.max(); top_missing = self.missing_summary.iloc[0]; top_num = self.numeric_drift.iloc[0]; top_cat = self.categorical_drift.iloc[0]
        items = [
            f"The training set has <b>{len(self.train):,}</b> rows and the test set has <b>{len(self.test):,}</b> rows.",
            f"The majority target class is <b>{html.escape(str(majority_class))}</b> at <b>{majority_share:.2f}%</b> of training rows.",
            f"The highest training missing rate is <b>{top_missing['train_missing_%']:.2f}%</b> in <b>{html.escape(str(top_missing['column']))}</b>.",
            f"The largest numeric train/test mean shift is in <b>{html.escape(str(top_num['column']))}</b> with absolute standardized mean difference <b>{abs(top_num['standardized_mean_diff']):.4f}</b>.",
            f"The largest categorical distribution shift is in <b>{html.escape(str(top_cat['column']))}</b> with total variation distance <b>{top_cat['total_variation_distance']:.4f}</b>.",
            "A practical first model should preserve missingness, encode categorical features, use stratified validation, and report per-class metrics.",
        ]
        return "<ul style='line-height:1.65;margin:0;padding-left:20px'>" + "".join(f"<li>{item}</li>" for item in items) + "</ul>"
NotebookDisplay.card("EDA takeaways", TakeawayWriter(config, train, test, missing_summary, numeric_drift, categorical_drift).render())
